In [1]:
!pip install -q giotto-tda

!pip install gudhi

!apt-get install -y python3-geopandas
!pip install geopandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.8/455.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 21.2 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  binfmt-support fonts-dejavu-core fonts-font-awesome fonts-lato fonts-lyx libclang-cpp11
  libffi-dev libimagequant0 liblbfgsb0 libllvm11 liblzo2-2 libpfm4 libraqm0 libspatialindex-c6
  libspatialindex-dev libspatialindex6 libxsimd-dev libz3-4 libz3-dev llvm-11 llvm-11-dev
  llvm-11-linker-tools llvm-11-runtime llvm-11-tools mailcap mime-support numba-doc
  python-babel-loca

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from gtda.homology import VietorisRipsPersistence
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Leer el archivo shapefile
shapefile_path = "Valencia_crime_hull.shp"
gdf = gpd.read_file(shapefile_path)

# Obtener los límites del shapefile
minx, miny, maxx, maxy = gdf.total_bounds

# Parámetros para generar los datos aleatorios
n = 94  # Número de puntos por dataset
m = 2    # Número de datasets a generar
j = 1000     # Número de repeticiones del experimento

# Función para generar puntos aleatorios dentro del shapefile
def generate_random_points_within_shapefile(num_points, gdf):
    points = []
    while len(points) < num_points:
        point = Point(np.random.uniform(minx, maxx), np.random.uniform(miny, maxy))
        if gdf.contains(point).any():
            points.append([point.x, point.y])  # Guardar coordenadas x, y
    return np.array(points)

# Función para calcular el diagrama de persistencia
def calculate_persistence_diagram(data, VR):
    scaler = StandardScaler()  # Escalar los datos
    data_scaled = scaler.fit_transform(data)  # Ajustar y transformar los datos
    diagrams = VR.fit_transform([data_scaled])
    return diagrams[0][:, :2]  # Tomar solo las columnas de nacimiento y muerte

# Instanciar Vietoris-Rips una vez
VR = VietorisRipsPersistence()

# Función para calcular la distancia de Wasserstein entre dos diagramas
def wasserstein_distance_between_diagrams(diag1, diag2):
    births1, deaths1 = diag1[:, 0], diag1[:, 1]
    births2, deaths2 = diag2[:, 0], diag2[:, 1]
    return (wasserstein_distance(births1, births2) +
            wasserstein_distance(deaths1, deaths2))

# Función para calcular la distancia Wasserstein media
def calculate_mean_wasserstein_distance(diagrams):
    num_diagrams = len(diagrams)
    total_wasserstein_distance = 0
    num_comparisons = 0

    for i in range(num_diagrams):
        for j in range(i + 1, num_diagrams):
            total_wasserstein_distance += wasserstein_distance_between_diagrams(diagrams[i], diagrams[j])
            num_comparisons += 1

    return total_wasserstein_distance / num_comparisons if num_comparisons > 0 else 0

# Experimento para calcular la media final de las distancias Wasserstein a través de j repeticiones
mean_wasserstein_across_experiments = []

for experiment in range(j):
    diagrams = []
    for _ in range(m):
        # Generar los puntos aleatorios dentro del shapefile
        random_points = generate_random_points_within_shapefile(n, gdf)

        # Calcular el diagrama de persistencia para estos puntos
        diagrams.append(calculate_persistence_diagram(random_points, VR))

    # Calcular la media de la distancia Wasserstein en este experimento
    mean_wasserstein = calculate_mean_wasserstein_distance(diagrams)
    mean_wasserstein_across_experiments.append(mean_wasserstein)
    print(f"Experiment {experiment+1}/{j}: Mean Wasserstein Distance = {mean_wasserstein}")

# Calcular la media final de las distancias Wasserstein
final_mean_wasserstein = np.mean(mean_wasserstein_across_experiments)
print(f"\nFinal Mean Wasserstein Distance after {j} experiments: {final_mean_wasserstein}")


Experiment 1/3: Mean Wasserstein Distance = 0.0003450417112339265
Experiment 2/3: Mean Wasserstein Distance = 0.00041846687521917734
Experiment 3/3: Mean Wasserstein Distance = 0.0003938839793011784

Final Mean Wasserstein Distance after 3 experiments: 0.00038579752191809404


In [3]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from gtda.homology import VietorisRipsPersistence
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Leer el archivo shapefile
shapefile_path = "Valencia_crime_hull.shp"
gdf = gpd.read_file(shapefile_path)

# Obtener los límites del shapefile
minx, miny, maxx, maxy = gdf.total_bounds

# Parámetros para generar los datos aleatorios
n = 94  # Número de puntos por dataset
m = 2    # Número de datasets a generar
j = 1000     # Número de repeticiones del experimento

# Función para generar puntos aleatorios dentro del shapefile, con una columna adicional 'day'
def generate_random_points_with_day(num_points, gdf):
    points = []
    while len(points) < num_points:
        point = Point(np.random.uniform(minx, maxx), np.random.uniform(miny, maxy))
        if gdf.contains(point).any():
            day = np.random.randint(1, 366)  # Valor aleatorio para 'day' entre 1 y 365
            points.append([point.x, point.y, day])  # Guardar coordenadas x, y y el valor de 'day'
    return np.array(points)

# Función para calcular el diagrama de persistencia en tres dimensiones
def calculate_persistence_diagram(data, VR):
    scaler = StandardScaler()  # Escalar los datos
    data_scaled = scaler.fit_transform(data)  # Ajustar y transformar los datos
    diagrams = VR.fit_transform([data_scaled])
    return diagrams[0][:, :2]  # Tomar solo las columnas de nacimiento y muerte

# Instanciar Vietoris-Rips para datos en 3D (dimensiones 0, 1 y 2)
VR = VietorisRipsPersistence(homology_dimensions=[0, 1, 2])

# Función para calcular la distancia de Wasserstein entre dos diagramas
def wasserstein_distance_between_diagrams(diag1, diag2):
    births1, deaths1 = diag1[:, 0], diag1[:, 1]
    births2, deaths2 = diag2[:, 0], diag2[:, 1]
    return (wasserstein_distance(births1, births2) +
            wasserstein_distance(deaths1, deaths2))

# Función para calcular la distancia Wasserstein media
def calculate_mean_wasserstein_distance(diagrams):
    num_diagrams = len(diagrams)
    total_wasserstein_distance = 0
    num_comparisons = 0

    for i in range(num_diagrams):
        for j in range(i + 1, num_diagrams):
            total_wasserstein_distance += wasserstein_distance_between_diagrams(diagrams[i], diagrams[j])
            num_comparisons += 1

    return total_wasserstein_distance / num_comparisons if num_comparisons > 0 else 0

# Experimento para calcular la media final de las distancias Wasserstein a través de j repeticiones
mean_wasserstein_across_experiments = []

for experiment in range(j):
    diagrams = []
    for _ in range(m):
        # Generar los puntos aleatorios dentro del shapefile con la columna 'day'
        random_points = generate_random_points_with_day(n, gdf)

        # Calcular el diagrama de persistencia para estos puntos
        diagrams.append(calculate_persistence_diagram(random_points, VR))

    # Calcular la media de la distancia Wasserstein en este experimento
    mean_wasserstein = calculate_mean_wasserstein_distance(diagrams)
    mean_wasserstein_across_experiments.append(mean_wasserstein)
    print(f"Experiment {experiment+1}/{j}: Mean Wasserstein Distance = {mean_wasserstein}")

# Calcular la media final de las distancias Wasserstein
final_mean_wasserstein = np.mean(mean_wasserstein_across_experiments)
print(f"\nFinal Mean Wasserstein Distance after {j} experiments: {final_mean_wasserstein}")


Experiment 1/1000: Mean Wasserstein Distance = 0.05591318481504082
Experiment 2/1000: Mean Wasserstein Distance = 0.1055090207397434
Experiment 3/1000: Mean Wasserstein Distance = 0.04754868765750268
Experiment 4/1000: Mean Wasserstein Distance = 0.06198521280393545
Experiment 5/1000: Mean Wasserstein Distance = 0.09773832463243343
Experiment 6/1000: Mean Wasserstein Distance = 0.08619116249964408
Experiment 7/1000: Mean Wasserstein Distance = 0.12684957286873957
Experiment 8/1000: Mean Wasserstein Distance = 0.06640408220078786
Experiment 9/1000: Mean Wasserstein Distance = 0.0546662455575464
Experiment 10/1000: Mean Wasserstein Distance = 0.11054339007834796
Experiment 11/1000: Mean Wasserstein Distance = 0.04540164681359529
Experiment 12/1000: Mean Wasserstein Distance = 0.06686739870031973
Experiment 13/1000: Mean Wasserstein Distance = 0.15415154238860979
Experiment 14/1000: Mean Wasserstein Distance = 0.15585597785812913
Experiment 15/1000: Mean Wasserstein Distance = 0.059912312